# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata

# Display high-level info
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata,'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will display all Record Sets in the dataset with their `@id`, `name`, and available fields with their `@id` and name.

In [ ]:
# List record sets, their @id, and fields
recordsets_info = []
for record_set in metadata.record_sets:
    print(f"RecordSet: {record_set.name} | @id: {record_set.id}")
    recordsets_info.append(record_set.id)
    for field in record_set.fields:
        print(f"    Field: {field.name} | @id: {field.id} | data type: {field.data_type}")
    print("-")
if not recordsets_info:
    print("No record sets found in the schema. Please check dataset documentation.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Below, we will demonstrate loading the first available `RecordSet` and its fields using the record set and field `@id`s from the overview.

In [ ]:
dataframes = {}
loaded_recordsets = []
# Handling case when no record sets exist:
if recordsets_info:
    for record_set_id in recordsets_info:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print(f"No records found for record set {record_set_id}.")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        loaded_recordsets.append(record_set_id)
        print(f"Loaded record set {record_set_id} with {len(df)} records. Columns available:")
        print(df.columns.tolist())
        # Only show first for brevity
        print(df.head())
        break  # Uncomment to load all record sets
else:
    dataframes = {}
    print("No record sets available to load.")
# For further analysis, select the first loaded record set
if loaded_recordsets:
    example_record_set_id = loaded_recordsets[0]
    df = dataframes[example_record_set_id]
else:
    example_record_set_id = None
    df = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering numeric fields, normalizing data, or grouping by categories. Choose a numeric field for demonstration based on the columns retrieved.

In [ ]:
import numpy as np

# For demonstration, try to find a numeric field automatically
if df is not None and len(df)>0:
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a categorical field for grouping (non-numeric, object type)
        group_candidates = df.select_dtypes(include=[object]).columns.tolist()
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping detected.")
    else:
        print("No numeric field detected for analysis.")
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. A histogram of the chosen numeric field and (if available) a group comparison barplot follows.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        order = df[group_field].value_counts().iloc[:10].index.tolist()
        sns.barplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(order)], order=order)
        plt.title(f"Mean {numeric_field} by {group_field} (top 10)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* We successfully loaded the Croissant metadata and inspected the available record sets and fields using their `@id` references.
* We loaded and explored a sample record set, performed basic EDA (filtering, normalization, grouping), and visualized one numeric field.
* For deeper analyses, examine more record sets and explore additional fields as defined by their Croissant `@id` in the schema.

> For more details, refer to the [mlcroissant documentation](https://mlcroissant.readthedocs.io/).